# rbench — Colab bootstrap

Spins up the retrieval-barrier testbed on a **Colab GPU runtime** and (optionally) exposes it to **local VS Code over Remote-SSH**. Nothing here touches your local hard drive — all installs, models, and datasets live on the Colab VM.

**First:** Runtime → Change runtime type → Hardware accelerator = **GPU**, then Save.

Run the cells top to bottom. Edit `REPO_URL` in cell 2 to your GitHub repo first.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Clone the repo
Edit `REPO_URL` to your repo (created in the next step of the chat).

In [ ]:
import os
REPO_URL = 'https://github.com/YOUR_USERNAME/rbench-retrieval-barrier.git'  # <-- EDIT ME
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd {REPO_DIR}
!git log --oneline -1

## 3. Install dependencies (onto the Colab VM, not your drive)

In [ ]:
!pip install -q -r requirements.txt
print('deps installed')

## 4. Run the Phase 1 smoke test on GPU
`--device auto` picks the GPU automatically.

In [ ]:
!python -m rbench.eval.sanity --device auto
print('\n--- hybrid + reranker ---\n')
!python -m rbench.eval.sanity --hybrid --device auto

## 5. (Optional) Persist results to Google Drive
Colab's disk is wiped when the session ends. Mount Drive and point outputs there so results survive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/rbench_results', exist_ok=True)
print('results dir on Drive: /content/drive/MyDrive/rbench_results')

## 6. (Optional) Expose this runtime to local VS Code over Remote-SSH

Run the cell below. It prints a Cloudflare hostname plus an SSH config block.

**On your laptop (one-time):**
1. Install [cloudflared](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/) and the VS Code **Remote - SSH** extension.
2. Paste the printed block into your `~/.ssh/config` (Windows: `C:\Users\<you>\.ssh\config`).
3. VS Code → Command Palette → *Remote-SSH: Connect to Host* → pick `google_colab_ssh`, enter the password below.

You are now editing on the Colab VM with GPU — local disk untouched. Note: the tunnel dies when the Colab session ends; re-run this cell to get a new one.

In [ ]:
!pip install -q colab-ssh
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password='CHANGE-ME-to-a-strong-password')